### Gathering Historical Fantasy Scores

In the cells below I will be gathering historical fantasy football players points for a specified season. I will first look at the 2025 season and perform transformations, cleaning, and EDA to determine if more data is needed and identify holes. This data is useful because we can use the historical points as a way to compare out models output performance as in the accuracy of our output.

This data is gathered from ESPN's default-leage endpoint api: [Fantasy API 2025](https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/2025)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import os
#root directory of project
root = Path.cwd().parent
data_path = root/"data"

In [2]:
def write_to_dict(df, name):
    '''Update dictionary csv files '''
    columns = df.columns.to_numpy()
    dict_ = pd.DataFrame(columns)
    try:
        dict_.to_csv(data_path/f"dicts/{name}_dictionary.csv")
    except Exception as e:
        print(f"failed to write dictionary with exception {e}")

def update_data(df, name):
    try:
        df.to_csv(data_path/f"{name}_final.csv")
    except Exception as e:
        print(f"failed to updata data: {e}")

In [3]:
## load fantasy historical data
weekly_df = pd.read_csv(data_path/"player_weeks_2025.csv")
#weekly_df.head()


In [4]:
# load player summary data
player_sum = pd.read_csv(data_path/"player_summary_2025.csv")
#player_sum.head()

In [5]:
write_to_dict(player_sum, "player_sum")


In [5]:
# load player log data
player_logs = pd.read_csv(data_path/"player_gamelogs_2025.csv")
#player_logs.head()


In [7]:
write_to_dict(player_logs, "player_logs")

In [8]:
display(weekly_df.shape)
display(player_sum.shape)
display(player_logs.shape)

(13958, 7)

(685, 104)

(7572, 70)

In [9]:
display(weekly_df["player_id"].nunique())
display(player_sum["player_id"].nunique())
display(player_logs["player_id"].nunique())

1088

685

685

In [6]:
## match our historical fantasy players with our other data
players_in_sum = player_sum["player_id"].unique()
players_set = set(players_in_sum)

players_in_logs = player_logs["player_id"].unique()
players_set_2 = set(players_in_logs)

assert players_set == players_set_2, "not same players"

In [7]:
# now that two of our df contain the same players, we only need to filter the historical fantasy data to contain the player_id as the other dataframes

weekly_df = weekly_df[weekly_df["player_id"].isin(players_set)]
weekly_df.shape

(11356, 7)

## cleaning missing values in projected points

There are some cases where a player was not set to play but ended up earning fantasy points and hence missing values for the projected points. So if a player has missing projected points and scored fantasy points, I am going to impute the projected points with either the median or mean of their fantasy points of that season, I will use the player summary dataframe to calculate these statistics and then determine which of the central tendencies I would like to use

In [8]:
mask = weekly_df[weekly_df["projected_points"].isna() & weekly_df["fantasy_points"].isna()]

# check why there are no columns with both no point and no projected points
no_proj = weekly_df[weekly_df["projected_points"].isna()]
no_pts = weekly_df[weekly_df["fantasy_points"].isna()]

display(mask.head())
display(no_proj.head())
display(no_pts.head())

print("length of no projected points:", len(no_proj))
print("length of no proj with week = 1:", len(no_proj[no_proj["week"] == 1]))
print(len(no_pts[no_pts["projected_points"] == 0]))

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
355,4601080,Raheim Sanders,RB,5,1,6.3,NaN
422,4430656,Myles Price,WR,16,1,0.0,NaN
450,4715355,Darius Cooper,WR,21,1,0.0,NaN
453,5160110,Jahdae Walker,WR,3,1,0.0,NaN
742,4876006,Zaire Mitchell-Paden,TE,18,1,0.0,NaN


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
144,4240657,Michael Carter,RB,22,1,NaN,0.0
149,4574716,Harrison Mevis,K,14,1,NaN,0.0
150,4430968,Kimani Vidal,RB,24,1,NaN,0.0
167,4034949,Eddy Pineiro,K,25,1,NaN,0.0
227,4569682,Audric Estime,RB,18,1,NaN,0.0


length of no projected points: 703
length of no proj with week = 1: 8
1064


In [9]:
# in player summary we already have average fantasy points so I will compute the mean 

weekly_df["med_proj_pts"]  = weekly_df.groupby("player_id")["projected_points"].transform("median")
#weekly_df.head()

In [10]:
weekly_df["projected_points"] = np.where(
    weekly_df["projected_points"].isna(),
    weekly_df["med_proj_pts"],
    weekly_df["projected_points"]
)

#weekly_df.head()

# Data Integration
From looking at the Data dictionaries, our summary and player logs contain the fantasy data already so I will Merge the logs with the ```weekly_df``` on player_id and week

In [11]:
#key columns are columns we merge on
key_cols = {"player_id","week"}

#find set of columns that overlap excluding key columns
overlapping_cols = (set(player_logs.columns) & set(weekly_df.columns)) - key_cols


#overlapping_cols

#left merge on key columns, drop overlapping columns
merged_df = pd.merge(player_logs, weekly_df.drop(columns=list(overlapping_cols)), on=list(key_cols), how="left")

#merged_df.head()

In [12]:
merged_df["team"].nunique()
merged_df["season_type"].nunique()

if merged_df["season_type"].nunique() == 1:
    merged_df = merged_df.drop(columns = ["pro_team_id", "season_type"])
#we have team name and team ID number
# merged_df.drop(columns = "pro_team_id")

#merged_df.head()

## Adding in Weather Data
Using ```nflready``` library, we can gather the weather location based on the value in the ```eventId``` column and use that to append corresponding weather data to the rows. 

In [13]:
import nflreadpy as nfl
import polars as pl


sched = nfl.load_schedules(seasons=2025)  
weather = sched.select([
    "espn", "game_id", "gameday", "gametime",
    "stadium", "roof", "surface", "temp", "wind"
])
weather = weather.to_pandas()
# your_df: your ESPN data with an event ID column
merged_df["eventId"] = merged_df["eventId"].astype(str)
df = merged_df.merge(weather, left_on="eventId", right_on="espn", how="left")
#df.head()

### Updating temperature data
We are going to impute the temperature missing values for domes to be the average temperature of a football dome stadium which happens to be around 70 farhenhiet.

In [14]:
# fixing roof data
df["roof"].unique()

df["roof"] = np.where(df["roof"] == "closed",
                      "dome", 
                      df["roof"])

df["roof"].head()

0    outdoors
1        dome
2        dome
3        dome
4        dome
Name: roof, dtype: str

In [15]:
df["roof"].unique()

<ArrowStringArray>
['outdoors', 'dome']
Length: 2, dtype: str

In [16]:
# impute mean indoor stadium temperature when theres a roof
df["temp"] = np.where(
    df["roof"] == "dome",
    70,
    df["temp"])

#there shouldnt be wind in a dome
df["wind"] = np.where(
    (df["wind"].isna()) & (df["roof"] == "dome"),
    0,
    df["wind"]
)


# finally, impute the outdoor wind values with the mean wind value outdoors
mean_wind = df[df["roof"] == "outdoors"]["wind"].mean()

df["wind"] = np.where(
    (df["wind"].isna()) & (df["roof"] == "outdoors"),
    mean_wind,
    df["wind"])



In [17]:
df["stadium"].unique()

<ArrowStringArray>
[                  'Soldier Field',               'U.S. Bank Stadium',
                      'Ford Field',                    'SoFi Stadium',
         'Lincoln Financial Field',                      'FedExField',
 'GEHA Field at Arrowhead Stadium',                  'Paycor Stadium',
                'M&T Bank Stadium',                   'Lambeau Field',
                  'Levi's Stadium',               'Lucas Oil Stadium',
             'FirstEnergy Stadium',              'State Farm Stadium',
                 'MetLife Stadium',                     'NRG Stadium',
           'Raymond James Stadium',         'Mercedes-Benz Superdome',
                     'Lumen Field',               'Hard Rock Stadium',
                   'New Era Field',                'Acrisure Stadium',
      'Empower Field at Mile High',           'Mercedes-Benz Stadium',
         'Bank of America Stadium',                  'Nissan Stadium',
                'Gillette Stadium',                    'AT

In [18]:
#fixing missing surface
df.loc[df["stadium"] == "SoFi Stadium","surface"] = "matrixturf"



## Adding in Location Data
I would also like to add Location data to this data set using the stadium name

In [19]:
h_a = {"@": "away", "vs": "home"}

df["atVs"] = df["atVs"].map(h_a)

#df.head()

In [20]:

stadium_locations = {
    'Soldier Field': 'Chicago,IL',
    'U.S. Bank Stadium': 'Minneapolis,MN',
    'Ford Field': 'Detroit,MI',
    'SoFi Stadium': 'Inglewood,CA',
    'Lincoln Financial Field': 'Philadelphia,PA',
    'FedExField': 'Landover,MD',
    'GEHA Field at Arrowhead Stadium': 'Kansas City,MO',
    'Paycor Stadium': 'Cincinnati,OH',
    'M&T Bank Stadium': 'Baltimore,MD',
    'Lambeau Field': 'Green Bay,WI',
    "Levi's Stadium": 'Santa Clara,CA',
    'Lucas Oil Stadium': 'Indianapolis,IN',
    'FirstEnergy Stadium': 'Cleveland,OH',
    'State Farm Stadium': 'Glendale,AZ',
    'MetLife Stadium': 'East Rutherford,NJ',
    'NRG Stadium': 'Houston,TX',
    'Raymond James Stadium': 'Tampa,FL',
    'Mercedes-Benz Superdome': 'New Orleans,LA',
    'Lumen Field': 'Seattle,WA',
    'Hard Rock Stadium': 'Miami Gardens,FL',
    'New Era Field': 'Orchard Park,NY',
    'Acrisure Stadium': 'Pittsburgh,PA',
    'Empower Field at Mile High': 'Denver,CO',
    'Mercedes-Benz Stadium': 'Atlanta,GA',
    'Bank of America Stadium': 'Charlotte,NC',
    'Nissan Stadium': 'Nashville,TN',
    'Gillette Stadium': 'Foxborough,MA',
    'AT&T Stadium': 'Arlington,TX',
    'Allegiant Stadium': 'Paradise,NV',
    'TIAA Bank Stadium': 'Jacksonville,FL'
}


df["location"] = df["stadium"].map(stadium_locations)

#redundant colunn

#df.head()

In [25]:
## update our data file
write_to_dict(df, "df")
update_data(df, "df")

## Adding features
We also need to consider how much playtime a player gets to and whether or not a player actually played as in their health status each game. We should also consider whether they are starters or second string players because that will impact how often they play and ultimately the number of fantasy points they are able to score.

In [22]:
df.columns

Index(['player_id', 'player', 'position', 'team', 'season', 'week', 'eventId',
       'date', 'opponent', 'atVs', 'result', 'score', 'rushingAttempts',
       'rushingYards', 'yardsPerRushAttempt', 'rushingTouchdowns',
       'longRushing', 'receptions', 'receivingTargets', 'receivingYards',
       'yardsPerReception', 'receivingTouchdowns', 'longReception', 'fumbles',
       'fumblesLost', 'fumblesForced', 'kicksBlocked', 'completions',
       'passingAttempts', 'passingYards', 'completionPct',
       'yardsPerPassAttempt', 'passingTouchdowns', 'interceptions',
       'longPassing', 'sacks', 'QBRating', 'adjQBR', 'longFieldGoalMade',
       'fieldGoalPct', 'fieldGoalsMadeYardsAverage', 'totalKickingPoints',
       'totalTackles', 'soloTackles', 'assistTackles', 'stuffs', 'stuffYards',
       'fumblesRecovered', 'interceptionYards', 'avgInterceptionYards',
       'interceptionTouchdowns', 'longInterception', 'passesDefended',
       'fieldGoalsMade1_19', 'fieldGoalAttempts1_19', 'field

In [23]:


df = df.copy()
df["date"] = pd.to_datetime(df["date"])

# Aggregate player statistics into team-game statistics.
team_games = (
    df.groupby(["season", "team", "game_id", "date"], as_index=False)
      .agg(
          team_pass_attempts=("passingAttempts", "sum"),
          team_passing_yards=("passingYards", "sum"),
          team_passing_tds=("passingTouchdowns", "sum"),
          team_rushing_attempts=("rushingAttempts", "sum"),
          team_rushing_yards=("rushingYards", "sum"),
      )
      .sort_values(["team", "date"])
)

# Calculate historical averages using prior games only.
features = [
    "team_pass_attempts",
    "team_passing_yards",
    "team_passing_tds",
    "team_rushing_attempts",
    "team_rushing_yards",
]

for col in features:
    team_games[f"{col}_last5"] = (
        team_games.groupby(["season", "team"])[col]
        .transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).mean()
        )
    )

In [24]:
team_games.head()

,season,team,game_id,date,team_pass_attempts,team_passing_yards,team_passing_tds,team_rushing_attempts,team_rushing_yards,team_pass_attempts_last5,team_passing_yards_last5,team_passing_tds_last5,team_rushing_attempts_last5,team_rushing_yards_last5
0,2025,ARI,2025_01_ARI_NO,2025-09-07 17:00:00+00:00,29.0,163.0,2.0,27.0,146.0,NaN,NaN,NaN,NaN,NaN
1,2025,ARI,2025_02_CAR_ARI,2025-09-14 20:05:00+00:00,25.0,220.0,1.0,22.0,82.0,29.000000,163.000000,2.000000,27.000000,146.000000
2,2025,ARI,2025_03_ARI_SF,2025-09-21 20:25:00+00:00,35.0,159.0,1.0,28.0,106.0,27.000000,191.500000,1.500000,24.500000,114.000000
3,2025,ARI,2025_04_SEA_ARI,2025-09-26 00:15:00+00:00,41.0,200.0,2.0,17.0,89.0,29.666667,180.666667,1.333333,25.666667,111.333333
4,2025,ARI,2025_05_TEN_ARI,2025-10-05 20:05:00+00:00,32.0,220.0,0.0,29.0,168.0,32.500000,185.500000,1.500000,23.500000,105.750000
